<a href="https://colab.research.google.com/github/GarimaChaubey/NLP-Natural-Language-Preprocessing-/blob/main/Qwen_Kaggle_IMDb_LoRA_Beginner_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beginner-Friendly Qwen Fine-Tuning with a Kaggle Dataset (LoRA)

**Goal:** Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` for movie-review sentiment classification using a real Kaggle dataset.

**Approach:** Use ready-made library calls. We will not write a custom training loop or implement a model from scratch.

**Dataset:** IMDb 50K Movie Reviews (`review`, `sentiment`) from Kaggle.

> **For a classroom demo:** We use a balanced subset of 2,000 reviews so training is faster. You can increase this later. A GPU (such as Colab T4) is strongly recommended.

### What you will do
1. Install the required libraries.
2. Download the public Kaggle dataset with `kagglehub`.
3. Create train, validation, and test splits.
4. Load Qwen and its tokenizer.
5. Check the model's predictions before fine-tuning.
6. Add LoRA adapters using PEFT.
7. Fine-tune using Hugging Face `Trainer`.
8. Test the fine-tuned model and save the adapter.


## 1. Select a GPU in Google Colab

In Colab, select **Runtime → Change runtime type → T4 GPU** (if available).

Then run the next cell. If you are using a local Jupyter environment, make sure the environment has a compatible PyTorch and CUDA setup.

In [ ]:
%pip -q install -U transformers peft datasets accelerate kagglehub pandas scikit-learn

## 2. Import the libraries

These are ready-made libraries:
- `kagglehub`: downloads a public Kaggle dataset.
- `pandas`: reads and inspects the CSV file.
- `scikit-learn`: creates stratified train/validation/test splits.
- `datasets`: converts data into Hugging Face Dataset objects.
- `transformers`: loads Qwen and runs training through `Trainer`.
- `peft`: adds LoRA adapters.

In [ ]:
import os
import glob
import torch
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model, TaskType

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected. Training on CPU will be much slower.")

## 3. Download the Kaggle IMDb dataset

We use KaggleHub so you do not need to manually upload the CSV file.

Dataset page: [IMDb 50K Movie Reviews on Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

The dataset contains movie reviews and their sentiment labels (`positive` or `negative`). Kaggle may ask you to sign in or accept the dataset's terms before downloading.

In [ ]:
dataset_path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)

print("Dataset downloaded to:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("CSV files found:", csv_files)

# Select the IMDb CSV file
csv_path = next(
    (path for path in csv_files if "imdb" in os.path.basename(path).lower()),
    csv_files[0]
)

df = pd.read_csv(csv_path)
print("CSV file:", csv_path)
print("Dataset shape:", df.shape)
df.head()

## 4. Prepare a small, balanced subset

The complete dataset has 50,000 reviews. To keep this first experiment manageable, select 1,000 positive and 1,000 negative reviews.

We then create:
- 80% training data
- 10% validation data
- 10% test data

**Why split the data?**
- Training set: used to update the LoRA adapter.
- Validation set: used to monitor training.
- Test set: kept separate for a final check.

In [ ]:
# Keep only the columns needed for this task
df = df[["review", "sentiment"]].dropna().copy()
df["sentiment"] = df["sentiment"].str.lower().str.strip()

# Select a balanced subset: 1,000 positive + 1,000 negative
balanced_df = (
    df.groupby("sentiment", group_keys=False)
      .sample(n=1000, random_state=42)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(balanced_df["sentiment"].value_counts())
print("Total selected reviews:", len(balanced_df))

In [ ]:
train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.20,
    random_state=42,
    stratify=balanced_df["sentiment"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["sentiment"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

## 5. Load Qwen and its tokenizer

`AutoTokenizer` converts text into token IDs. `AutoModelForCausalLM` loads the pre-trained decoder-only language model.

This downloads the model checkpoint the first time it is run.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded:", model_name)
print("Parameters (millions):", round(
    sum(p.numel() for p in model.parameters()) / 1e6, 2
))

## 6. Test Qwen before fine-tuning

This is the baseline. We ask the original model to classify one review before training any LoRA adapter.

In [ ]:
def ask_qwen(review, max_new_tokens=8):
    messages = [
        {
            "role": "system",
            "content": "Classify the sentiment of the movie review. Reply with only Positive or Negative."
        },
        {
            "role": "user",
            "content": f"Review: {review}"
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


sample_review = test_df.iloc[0]["review"]
print("Review:", sample_review[:400])
print("Before fine-tuning:", ask_qwen(sample_review))

## 7. Convert each labelled review into a Qwen chat example

For supervised fine-tuning, we provide:
- a system instruction,
- the review as the user message,
- the correct sentiment as the assistant answer.

The model learns to generate the answer (`Positive` or `Negative`) from examples.

In [ ]:
def format_example(review, label):
    label = label.capitalize()

    messages = [
        {
            "role": "system",
            "content": "Classify the sentiment of the movie review. Reply with only Positive or Negative."
        },
        {
            "role": "user",
            "content": f"Review: {review}"
        },
        {
            "role": "assistant",
            "content": label
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )


train_texts = [format_example(row.review, row.sentiment)
               for row in train_df.itertuples()]
val_texts = [format_example(row.review, row.sentiment)
             for row in val_df.itertuples()]
test_texts = [format_example(row.review, row.sentiment)
              for row in test_df.itertuples()]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})
test_dataset = Dataset.from_dict({"text": test_texts})

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("\nOne formatted example:\n")
print(train_dataset[0]["text"])

## 8. Add LoRA adapters

LoRA does not update all the original Qwen weights. It freezes the base model and trains small adapter matrices.

- `r`: LoRA rank, which controls adapter size.
- `lora_alpha`: scaling factor.
- `lora_dropout`: dropout applied in the LoRA layers.
- `target_modules`: attention projection layers where adapters are added.

For this first run, keep these values unchanged. You can experiment with them later.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Disable the KV cache during training
model.config.use_cache = False

## 9. Tokenize the dataset

Tokenization converts each formatted conversation into token IDs.

We use a maximum sequence length of 512 tokens to keep this introductory experiment manageable. The data collator prepares batches and creates language-model labels.

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(tokenized_train)


## 10. Fine-tune using Hugging Face Trainer

`Trainer` manages the training loop for us. We only specify the training settings and datasets.

This is the actual fine-tuning step. The LoRA adapter weights are updated; the original Qwen weights remain frozen.

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-imdb-lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

trainer.train()

## 11. Test after fine-tuning

Now ask the adapted model the same question. Compare its answer with the baseline result from Section 6.

A single example is only a quick demonstration. For a meaningful result, evaluate the entire held-out test set.

In [ ]:
print("Review:", sample_review[:400])
print("After fine-tuning:", ask_qwen(sample_review))

## 12. Simple test-set accuracy

We generate a label for each held-out review and compare it with the true Kaggle label.

This is a simple evaluation helper for the classroom demo. It may take a few minutes depending on your GPU.

In [ ]:
def predict_label(review):
    prediction = ask_qwen(review, max_new_tokens=8).lower()

    # Extract the first expected label from the generated answer
    if "positive" in prediction:
        return "positive"
    if "negative" in prediction:
        return "negative"
    return "unknown"


# Use a smaller test sample first to keep the demo quick.
evaluation_df = test_df.sample(n=min(50, len(test_df)), random_state=42).copy()

evaluation_df["prediction"] = evaluation_df["review"].apply(predict_label)
accuracy = (evaluation_df["prediction"] == evaluation_df["sentiment"]).mean()

print("Evaluated examples:", len(evaluation_df))
print(f"Test-sample accuracy: {accuracy:.2%}")
print(evaluation_df[["sentiment", "prediction"]].head(10))

## 13. Save the fine-tuned LoRA adapter

The adapter is saved separately from the original base model. To use it later, load the same base model and attach this adapter.

In [ ]:
adapter_path = "./qwen-imdb-lora-adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved to:", adapter_path)

## 14. What did we learn?

| Term | Meaning in this notebook |
|---|---|
| Kaggle dataset | Real labelled IMDb movie reviews |
| Pre-trained model | Qwen already learned general language patterns |
| Fine-tuning | Adapt Qwen using labelled movie-review examples |
| LoRA / PEFT | Freeze the base model and train small adapter weights |
| Training set | Examples used to update the adapter |
| Validation set | Used to monitor training |
| Test set | Held-out examples used for a final check |
| Inference | Asking the fine-tuned model to classify a new review |

### Important limitations
- We trained on only 2,000 reviews for a quick demonstration.
- The test accuracy is computed on a 50-review sample, so it is only a rough classroom check.
- This simple notebook calculates language-model loss over the full formatted conversation. A more advanced SFT pipeline can mask prompt tokens and calculate loss only on the assistant answer.
- A larger model, more data, better hyperparameters, and repeated evaluation may be needed for a stronger result.

### Try these beginner experiments
1. Change `num_train_epochs` from `1` to `2`.
2. Change LoRA rank `r` from `8` to `4` or `16`.
3. Change `learning_rate` from `2e-4` to `1e-4`.
4. Increase the balanced subset from 1,000 to 2,500 reviews per class.
5. Compare the validation loss and test-sample accuracy.